# Final library formatting for GNPS2

Combines the per-subclass filtered MGFs and annotation tables into the six libraries, maps the new scan numbers, adds instrument information, flags negative deltas as potential ISF, fills the GNPS2 batch template and writes the final MGFs (header lines reduced to PEPMASS, CHARGE and SCANS).

In [ ]:
import os

# Root of the local project folder (the one holding Falcon/, Queries_final_mgf/,
# final_mgfs_library_generation/, ...). Edit this single line to run elsewhere.
PROJECT_DIR = '/media/dorrestein/Helena2/Bile_acids_update'

In [ ]:
import pandas as pd

## Combine MGFs

In [ ]:
import re
import glob

def combine_mgf(input_files, output_file, start=1):
    scan = start
    with open(output_file, "w") as out:
        for path in input_files:
            with open(path) as f:
                for line in f:
                    stripped = line.strip()
                    if stripped.startswith("SCANS="):
                        out.write(f"SCANS={scan}\n")
                    elif stripped == "END IONS":
                        out.write(line)
                        scan += 1          # advance only after a full spectrum
                    else:
                        out.write(line)
    print(f"Wrote {scan - start} spectra to {output_file} (SCANS {start}..{scan-1})")

In [ ]:
c24_input_files = [
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_mgf/C24_nonhydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_mgf/C24_monohydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_mgf/C24_dihydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_mgf/C24_trihydroxy_combined_SpecReBoot_filtered_MN_and_entropy.mgf',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_mgf/C24_tetrahydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_mgf/C24_pentahydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
]

c23_input_files = [
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_mgf/C23_monohydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_mgf/C23_dihydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_mgf/C23_trihydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_mgf/C23_tetrahydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_mgf/C23_pentahydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
]

c27_input_files = [
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_mgf/C27_monohydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_mgf/C27_dihydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_mgf/C27_trihydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_mgf/C27_tetrahydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_mgf/C27_pentahydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
]

c26_input_files = [
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_mgf/Aceto_dihydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
]

c24_alkylamine_input_files = [
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_mgf/C24_alkylamine_monohydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_mgf/C24_alkylamine_dihydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_mgf/C24_alkylamine_trihydroxy_combined_SpecReBoot_filtered_MN_and_entropy.mgf',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_mgf/C24_alkylamine_tetrahydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_mgf/C24_alkylamine_pentahydroxy_SpecReBoot_filtered_MN_and_entropy.mgf',
]

c28_alcohol_input_files = [
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_mgf/C28_alcohol_1OH_falcon_eps01_minsamples2_scan.mgf',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_mgf/C28_alcohol_2OH_SpecReBoot_filtered_MN_and_entropy.mgf',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_mgf/C28_alcohol_3OH_SpecReBoot_filtered_MN_and_entropy.mgf', 
]

In [ ]:
combine_mgf(c24_input_files, f"{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C24_library.mgf")
combine_mgf(c23_input_files, f"{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C23_library.mgf")
combine_mgf(c27_input_files, f"{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C27_library.mgf")
combine_mgf(c26_input_files, f"{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C26_library.mgf")
combine_mgf(c24_alkylamine_input_files, f"{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C24_alkylamine_library.mgf")
combine_mgf(c28_alcohol_input_files, f"{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C28_alcohol_library.mgf")

## Combine TSVs

In [ ]:
### now we need the ANNOTATION TABLES to also be combined, then map the scan numbers to the new combined mgfs

c24_annotation_table = [
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_tsv_annotation/c24_nonhydroxy_buddy_sirius_merged_final_annotation.tsv',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_tsv_annotation/c24_monohydroxy_buddy_sirius_merged_final_annotation.tsv',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_tsv_annotation/c24_dihydroxy_buddy_sirius_merged_final_annotation.tsv',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_tsv_annotation/c24_trihydroxy_combined_buddy_sirius_merged_final_annotation.tsv',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_tsv_annotation/c24_tetrahydroxy_buddy_sirius_merged_final_annotation.tsv',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_tsv_annotation/c24_pentahydroxy_buddy_sirius_merged_final_annotation.tsv',
]

c23_annotation_table = [
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_tsv_annotation/C23_monohydroxy_buddy_sirius_merged_final_annotation.tsv',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_tsv_annotation/C23_dihydroxy_buddy_sirius_merged_final_annotation.tsv',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_tsv_annotation/C23_trihydroxy_buddy_sirius_merged_final_annotation.tsv',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_tsv_annotation/C23_tetrahydroxy_buddy_sirius_merged_final_annotation.tsv',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_tsv_annotation/C23_pentahydroxy_buddy_sirius_merged_final_annotation.tsv',
]

c27_annotation_table = [
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_tsv_annotation/C27_monohydroxy_buddy_sirius_merged_final_annotation.tsv',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_tsv_annotation/C27_dihydroxy_buddy_sirius_merged_final_annotation.tsv',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_tsv_annotation/C27_trihydroxy_buddy_sirius_merged_final_annotation.tsv',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_tsv_annotation/C27_tetrahydroxy_buddy_sirius_merged_final_annotation.tsv',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_tsv_annotation/C27_pentahydroxy_buddy_sirius_merged_final_annotation.tsv',
]

c26_annotation_table = [
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_tsv_annotation/Aceto_dihydroxy_buddy_sirius_merged_final_annotation.tsv',
]

c24_alkylamine_annotation_table = [
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_tsv_annotation/C24_alkylamine_monohydroxy_buddy_sirius_merged_final_annotation.tsv',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_tsv_annotation/C24_alkylamine_dihydroxy_buddy_sirius_merged_final_annotation.tsv',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_tsv_annotation/C24_alkylamine_trihydroxy_combined_buddy_sirius_merged_final_annotation.tsv',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_tsv_annotation/C24_alkylamine_tetrahydroxy_buddy_sirius_merged_final_annotation.tsv',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_tsv_annotation/C24_alkylamine_pentahydroxy_buddy_sirius_merged_final_annotation.tsv',
]

c28_alcohol_annotation_table = [
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_tsv_annotation/C28_alcohol_1OH_buddy_sirius_merged_final_annotation.tsv',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_tsv_annotation/C28_alcohol_2OH_buddy_sirius_merged_final_annotation.tsv',
    f'{PROJECT_DIR}/final_mgfs_library_generation/individual_files_tsv_annotation/C28_alcohol_3OH_buddy_sirius_merged_final_annotation.tsv',
]

In [ ]:
import pandas as pd

def combine_tsv(input_files, output_file, sep="\t"):
    dfs = [pd.read_csv(path, sep=sep) for path in input_files]
    combined = pd.concat(dfs, ignore_index=True)
    combined.to_csv(output_file, sep=sep, index=False)
    print(f"Combined {len(input_files)} tables -> {output_file}  ({len(combined)} rows)")
    return combined

In [ ]:
out_dir = f"{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS"

combine_tsv(c24_annotation_table,            f"{out_dir}/C24_annotation_combined.tsv")
combine_tsv(c23_annotation_table,            f"{out_dir}/C23_annotation_combined.tsv")
combine_tsv(c27_annotation_table,            f"{out_dir}/C27_annotation_combined.tsv")
combine_tsv(c26_annotation_table,            f"{out_dir}/C26_annotation_combined.tsv")
combine_tsv(c24_alkylamine_annotation_table, f"{out_dir}/C24_alkylamine_annotation_combined.tsv")
combine_tsv(c28_alcohol_annotation_table,    f"{out_dir}/C28_alcohol_annotation_combined.tsv")

In [ ]:
def map_scans(mgf_file, annotation_file, output_file):
    scans = []
    title = []
    with open(mgf_file, 'rb') as fin:
        for line in fin.readlines():
            decoded_line = line.decode('utf-8', errors='ignore')
            if decoded_line.startswith('SCANS='):
                scans.append(decoded_line.strip().split('=')[1])
            elif decoded_line.startswith('TITLE='):
                title.append(decoded_line.strip().split('=')[1])

    df = pd.DataFrame()
    df['scan_final_mgf'] = scans
    df['identifier'] = title

    annotation_table = pd.read_csv(annotation_file, sep='\t')
    annotation_table_merged = pd.merge(annotation_table, df, on='identifier', how='left')
    annotation_table_merged.to_csv(output_file, sep='\t', index=False)

    n_missing = annotation_table_merged['scan_final_mgf'].isna().sum()
    print(f"{output_file}  ->  {len(annotation_table_merged)} rows, {n_missing} unmapped")

In [ ]:
base = f"{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS"

pairs = [
    (f"{base}/C23_library.mgf",            f"{base}/C23_annotation_combined.tsv"),
    (f"{base}/C24_alkylamine_library.mgf", f"{base}/C24_alkylamine_annotation_combined.tsv"),
    (f"{base}/C24_library.mgf",            f"{base}/C24_annotation_combined.tsv"),
    (f"{base}/C26_library.mgf",            f"{base}/C26_annotation_combined.tsv"),
    (f"{base}/C27_library.mgf",            f"{base}/C27_annotation_combined.tsv"),
    (f"{base}/C28_alcohol_library.mgf",    f"{base}/C28_alcohol_annotation_combined.tsv"),
]

for mgf_file, annotation_file in pairs:
    output_file = annotation_file.replace("_combined.tsv", "_combined_scan.tsv")
    map_scans(mgf_file, annotation_file, output_file)

## Get instrument type

In [ ]:
# now we need to get the instrument type from the scans... get back to the annotationt table pre-falcon 

pre_falcon = {
    'C24_nonhydroxy': f'{PROJECT_DIR}/Queries_final_mgf/C24_final/Nonhydroxy/download_mgfs/final_mgf/C24_Nonhydroxy_input_falcon_charge_annotation_table.tsv',
    'C24_monohydroxy': f'{PROJECT_DIR}/Queries_final_mgf/C24_final/Monohydroxy/download_mgfs/final_mgf/C24_Monohydroxy_input_falcon_charge_annotation_table.tsv',
    'C24_dihydroxy': f'{PROJECT_DIR}/Queries_final_mgf/C24_final/Dihydroxy/download_mgfs/final_mgf/C24_dihydroxy_input_falcon_charge_annotation_table.tsv',
    'C24_trihydroxy_part1': f'{PROJECT_DIR}/Queries_final_mgf/C24_final/Trihydroxy/download_mgfs/final_mgf/C24_Trihydroxy_input_falcon_charge_part1_annotation_table.tsv',
    'C24_trihydroxy_part2': f'{PROJECT_DIR}/Queries_final_mgf/C24_final/Trihydroxy/download_mgfs/final_mgf/C24_Trihydroxy_input_falcon_charge_part2_annotation_table.tsv',
    'C24_trihydroxy_part3': f'{PROJECT_DIR}/Queries_final_mgf/C24_final/Trihydroxy/download_mgfs/final_mgf/C24_Trihydroxy_input_falcon_charge_part3_annotation_table.tsv',
    'C24_tetrahydroxy': f'{PROJECT_DIR}/Queries_final_mgf/C24_final/Tetrahydroxy/download_mgfs/final_mgf/C24_Tetrahydroxy_input_falcon_charge_annotation_table.tsv',
    'C24_pentahydroxy': f'{PROJECT_DIR}/Queries_final_mgf/C24_final/Pentahydroxy/download_mgfs/final_mgf/C24_Pentahydroxy_input_falcon_charge_annotation_table.tsv',

    "C23_monohydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C23/C23_monohydroxy/download_mgfs/final_mgf/C23_Monohydroxy_input_falcon_charge_annotation_table.tsv',
    "C23_dihydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C23/C23_dihydroxy/download_mgfs/final_mgf/C23_dihydroxy_input_falcon_charge_annotation_table.tsv',
    "C23_trihydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C23/C23_trihydroxy/download_mgfs/final_mgf/C23_trihydroxy_input_falcon_charge_annotation_table.tsv',
    "C23_tetrahydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C23/C23_tetrahydroxy/download_mgfs/final_mgf/C23_tetrahydroxy_input_falcon_charge_annotation_table.tsv',
    "C23_pentahydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C23/C23_pentahydroxy/download_mgfs/final_mgf/C23_pentahydroxy_input_falcon_charge_annotation_table.tsv',

    "C27_monohydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C27/C27_monohydroxy/download_mgfs/final_mgf/C27_monohydroxy_input_falcon_charge_annotation_table.tsv',
    "C27_dihydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C27/C27_dihydroxy/download_mgfs/final_mgf/C27_dihydroxy_input_falcon_charge_annotation_table.tsv',
    "C27_trihydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C27/C27_trihydroxy/download_mgfs/final_mgf/C27_trihydroxy_input_falcon_charge_annotation_table.tsv',
    "C27_tetrahydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C27/C27_tetrahydroxy/download_mgfs/final_mgf/C27_tetrahydroxy_input_falcon_charge_annotation_table.tsv',
    "C27_pentahydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C27/C27_pentahydroxy/download_mgfs/final_mgf/C27_pentahydroxy_input_falcon_charge_annotation_table.tsv',

    "Aceto_dihydroxy": f'{PROJECT_DIR}/Queries_final_mgf/Aceto/Dihydroxy/download_mgfs/final_mgf/Aceto_dihydroxy_input_falcon_charge_renumbered_annotation_table.tsv',

    "C24_alkylamine_monohydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_monohydroxy/download_mgfs/final_mgf/C24_alkylamine_monohydroxy_input_falcon_charge_annotation_table.tsv',
    "C24_alkylamine_dihydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_dihydroxy/download_mgfs/final_mgf/C24_alkylamine_dihydroxy_input_falcon_charge_annotation_table.tsv',
    "C24_alkylamine_trihydroxy_part1": f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/download_mgfs/final_mgf/C24_alkylamine_trihydroxy_input_falcon_charge_part1_annotation_table.tsv',
    "C24_alkylamine_trihydroxy_part2": f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/download_mgfs/final_mgf/C24_alkylamine_trihydroxy_input_falcon_charge_part2_annotation_table.tsv',
    "C24_alkylamine_trihydroxy_part3": f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/download_mgfs/final_mgf/C24_alkylamine_trihydroxy_input_falcon_charge_part3_annotation_table.tsv',
    "C24_alkylamine_trihydroxy_part4": f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_trihydroxy/download_mgfs/final_mgf/C24_alkylamine_trihydroxy_input_falcon_charge_part4_annotation_table.tsv',
    "C24_alkylamine_tetrahydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_tetrahydroxy/download_mgfs/final_mgf/C24_alkylamine_tetrahydroxy_input_falcon_charge_annotation_table.tsv',
    "C24_alkylamine_pentahydroxy": f'{PROJECT_DIR}/Queries_final_mgf/C24_alkylamine/C24_alkylamine_pentahydroxy/download_mgfs/final_mgf/C24_alkylamine_pentahydroxy_input_falcon_charge_annotation_table.tsv',

    "C28_alcohol_1OH": f'{PROJECT_DIR}/Queries_final_mgf/C28_alcohol/One_OH/download_mgfs/final_mgf/C28_alcohol_1OH_input_falcon_351_C28O1_charge_annotation_table.tsv',
    "C28_alcohol_2OH": f'{PROJECT_DIR}/Queries_final_mgf/C28_alcohol/Two_OH/download_mgfs/final_mgf/C28_alcohol_2OH_input_falcon_367_C28O2_charge_annotation_table.tsv',
    "C28_alcohol_3OH": f'{PROJECT_DIR}/Queries_final_mgf/C28_alcohol/Three_OH/download_mgfs/final_mgf/C28_alcohol_3OH_input_falcon_383_C28O3_charge_annotation_table.tsv',
}

#create an identifier column and combine all of them with another column specifying from where the rows came from

import os
import pandas as pd

all_dfs = []

for name, path in pre_falcon.items():
    df = pd.read_csv(path, sep='\t')
    df.columns = df.columns.str.strip()

    # filename without the '_annotation_table.tsv' part
    file_stem = os.path.basename(path).replace('_annotation_table.tsv', '')

    # 1) build the identifier column
    df['identifier'] = ('mzspec:USI000000:' + file_stem
                        + ':scan:' + df['scan_combined_mgf'].astype(str))

    # 2) tag with the dictionary key (name of the dataframe)
    df['source'] = name

    all_dfs.append(df)

combined = pd.concat(all_dfs, ignore_index=True)
print(f"Combined {len(all_dfs)} tables -> {len(combined)} rows")

# make some small adjusments for successful merge after
combined['identifier'] = combined['identifier'].str.replace(
    'C24_Monohydroxy_input_falcon_charge',
    'C24_monohydroxy_input_falcon_charge',
    regex=False
)

combined['identifier'] = combined['identifier'].str.replace(
    'C24_Pentahydroxy_input_falcon_charge',
    'C24_pentahydroxy_input_falcon_charge',
    regex=False
)

combined['identifier'] = combined['identifier'].str.replace(
    'C24_Tetrahydroxy_input_falcon_charge',
    'C24_tetrahydroxy_input_falcon_charge',
    regex=False
)

In [ ]:
library_tsv = [
    f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C23_annotation_combined_scan.tsv',
    f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C24_alkylamine_annotation_combined_scan.tsv',
    f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C24_annotation_combined_scan.tsv',
    f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C26_annotation_combined_scan.tsv',
    f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C27_annotation_combined_scan.tsv',
    f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C28_alcohol_annotation_combined_scan.tsv'
]

#combine all the annotation tables
combined_libraries = combine_tsv(library_tsv, f"{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/combined_library_annotation_scan.tsv")

#and combine with the original scan information
combined_libraries_lookup = combined_libraries.copy()
combined_lookup = combined[['identifier', 'origin_repository', 'original_path']].copy()
combined_libraries_lookup['merge_identifier'] = combined_libraries_lookup['identifier'].str.replace('Aceto_dihydroxy_input_falcon_charge_renumbered', 'Aceto_dihydroxy_input_falcon_charge', regex=False)
combined_lookup['merge_identifier'] = combined_lookup['identifier'].str.replace('Aceto_dihydroxy_input_falcon_charge_renumbered', 'Aceto_dihydroxy_input_falcon_charge', regex=False)
final_combined = pd.merge(combined_libraries_lookup, combined_lookup[['merge_identifier', 'origin_repository', 'original_path']], on='merge_identifier', how='left')
final_combined = final_combined.drop(columns=['merge_identifier'])
final_combined['dataset'] = final_combined['original_path'].str.split('/').str[2]  # extract the dataset name from the path

In [ ]:
#fill out what we can from MassIVE
final_combined['origin_repository'] = final_combined['origin_repository'].str.strip()

final_combined['INSTRUMENT'] = final_combined['origin_repository'].map(
    {'massive_qe': 'Orbitrap', 'massive_orbitrap': 'Orbitrap', 'massive_qtof': 'qTof'}
)

#then merge with the instrument curation manually done
instruments = pd.read_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/instruments_ba_update.txt', sep='\t')

# bring in the curated instrument column under a temp name, keyed on 'dataset'
final_combined = final_combined.merge(
    instruments[['dataset', 'INSTRUMENT']].rename(columns={'INSTRUMENT': 'INSTRUMENT_curated'}),
    on='dataset',
    how='left'
)

# fill ONLY the empty cells; prepopulated Orbitrap/qTof rows stay untouched
final_combined['INSTRUMENT'] = final_combined['INSTRUMENT'].fillna(final_combined['INSTRUMENT_curated'])

# drop the temp column
final_combined = final_combined.drop(columns='INSTRUMENT_curated')

In [ ]:
import numpy as np

#specify that negative deltas can be ISF
final_combined['annotation_isf'] = np.where(
    final_combined['delta_mass'] < 0,
    final_combined['annotation'] + '; potential ISF',
    final_combined['annotation']
)

#export
final_combined.to_csv(f"{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/combined_library_annotation_scan_instrument.tsv", sep='\t', index=False)

# Template

### TSV

In [ ]:
df = pd.read_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/combined_library_annotation_scan_instrument.tsv', sep='\t')
c23_library = df[df['annotation'].str.contains('C23_')]
c24_library = df[df['annotation'].str.contains('c24_')]
c24_alkylamine_library = df[df['annotation'].str.contains('C24_alkylamine')]
c26_library = df[df['annotation'].str.contains('Aceto_dihydroxy')]
c27_library = df[df['annotation'].str.contains('C27_')]
c28_alcohol_library = df[df['annotation'].str.contains('C28_alcohol_')]

In [ ]:
columns_template=['FILENAME','SEQ','COMPOUND_NAME','MOLECULEMASS','INSTRUMENT', 'IONSOURCE','EXTRACTSCAN', 'SMILES',
                  'INCHI', 'INCHIAUX', 'CHARGE', 'IONMODE', 'PUBMED', 'ACQUISITION', 'EXACTMASS', 'DATACOLLECTOR',
                  'ADDUCT', 'INTEREST', 'LIBQUALITY', 'GENUS', 'SPECIES', 'STRAIN', 'CASNUMBER', 'PI']

def fill_template(library, filename):
    template_df = pd.DataFrame(index=library.index)

    template_df['FILENAME'] = filename
    template_df['SEQ'] = '*..*'
    template_df['COMPOUND_NAME'] = library['annotation_isf']
    template_df['MOLECULEMASS'] = library['pepmass']
    template_df['INSTRUMENT'] = library['INSTRUMENT']
    template_df['IONSOURCE'] = 'LC-ESI'
    template_df['EXTRACTSCAN'] = library['scan_final_mgf']
    template_df[['SMILES', 'INCHI', 'INCHIAUX', 'PUBMED', 'INTEREST', 'GENUS', 'SPECIES', 'STRAIN', 'CASNUMBER']] = 'N/A'
    template_df['CHARGE'] = '0'
    template_df['IONMODE'] = 'Positive'
    template_df['ACQUISITION'] = 'Crude'
    template_df['EXACTMASS'] = library['pepmass']
    template_df['DATACOLLECTOR'] = 'helenamrusso'
    template_df['ADDUCT'] = 'M+H'
    template_df['LIBQUALITY'] = '4'
    template_df['PI'] = 'Pieter Dorrestein'

    return template_df

c23_template = fill_template(c23_library, 'C23_library_final.mgf')
c24_template = fill_template(c24_library, 'C24_library_final.mgf')
c24_alkylamine_template = fill_template(c24_alkylamine_library, 'C24_alkylamine_library_final.mgf')
c26_template = fill_template(c26_library, 'C26_library_final.mgf')
c27_template = fill_template(c27_library, 'C27_library_final.mgf')
c28_alcohol_template = fill_template(c28_alcohol_library, 'C28_alcohol_library_final.mgf')

In [ ]:
# export all the templates
c23_template.to_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C23_library_final_annotation.tsv', sep='\t', index=False)
c24_template.to_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C24_library_final_annotation.tsv', sep='\t', index=False)
c24_alkylamine_template.to_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C24_alkylamine_library_final_annotation.tsv', sep='\t', index=False)
c26_template.to_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C26_library_final_annotation.tsv', sep='\t', index=False)
c27_template.to_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C27_library_final_annotation.tsv', sep='\t', index=False)
c28_alcohol_template.to_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C28_alcohol_library_final_annotation.tsv', sep='\t', index=False)

In [ ]:
#final adjustments

####C28_alcohol
c28_alcohol_library = pd.read_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C28_alcohol_library_final_annotation.tsv', sep='\t')

import re

def clean_name(name):
    m = re.search(
        r'^(.*?)\s*\(sirius formula:\s*(.*?);\s*buddy formula:\s*(.*?)\)\s*$',
        name
    )
    if not m:
        return name  # no formula parenthetical, leave unchanged

    base, sirius, buddy = m.group(1), m.group(2).strip(), m.group(3).strip()
    parts = []
    if sirius != 'N/A':
        parts.append(f'sirius formula: {sirius}')
    if buddy != 'N/A':
        parts.append(f'buddy formula: {buddy}')

    return f'{base} ({"; ".join(parts)})' if parts else base

c28_alcohol_library['COMPOUND_NAME'] = (
    c28_alcohol_library['COMPOUND_NAME'].apply(clean_name)
)

c28_alcohol_library.to_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C28_alcohol_library_final_annotation.tsv', sep='\t', index=False)


####C24_alkylamine
c24_alkamine_library = pd.read_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C24_alkylamine_library_final_annotation.tsv', sep='\t')

c24_alkamine_library['COMPOUND_NAME'] = (
    c24_alkamine_library['COMPOUND_NAME'].str.replace('alkylamine', 'alkamine', regex=False)
)

c24_alkamine_library.to_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C24_alkylamine_library_final_annotation.tsv', sep='\t')


####C26_aceto
c26_dihydroxy_library = pd.read_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C26_library_final_annotation.tsv', sep='\t')

c26_dihydroxy_library['COMPOUND_NAME'] = (
    c26_dihydroxy_library['COMPOUND_NAME'].str.replace('Aceto_', 'C26_', regex=False)
)

c26_dihydroxy_library.to_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C26_library_final_annotation.tsv', sep='\t')


####C24
c24_library = pd.read_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C24_library_final_annotation.tsv', sep='\t')

c24_library['COMPOUND_NAME'] = (
    c24_library['COMPOUND_NAME'].str.replace('c24_', 'C24_', regex=False)
)

c24_library.to_csv(f'{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C24_library_final_annotation.tsv', sep='\t')

### MGF

In [ ]:
def filter_mgf(input_file, output_file):
    keep_prefixes = ('PEPMASS', 'CHARGE', 'SCANS')
    with open(input_file) as fin, open(output_file, 'w') as out:
        for line in fin:
            stripped = line.strip()
            if stripped == 'BEGIN IONS' or stripped == 'END IONS':
                out.write(line)
            elif stripped.startswith(keep_prefixes):
                out.write(line)
            elif '=' in stripped:
                continue   # drop other header lines (TITLE, RTINSECONDS, CLUSTER, etc.)
            else:
                out.write(line)   # ion peak lines (no '=') are kept
    print(f"Wrote filtered MGF to {output_file}")

In [ ]:
filter_mgf(f"{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C23_library.mgf", 
           f"{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C23_library_final.mgf")
filter_mgf(f"{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C24_library.mgf", 
           f"{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C24_library_final.mgf")
filter_mgf(f"{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C24_alkylamine_library.mgf", 
           f"{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C24_alkylamine_library_final.mgf")
filter_mgf(f"{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C26_library.mgf", 
           f"{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C26_library_final.mgf")
filter_mgf(f"{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C27_library.mgf", 
           f"{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C27_library_final.mgf")
filter_mgf(f"{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/C28_alcohol_library.mgf", 
           f"{PROJECT_DIR}/final_mgfs_library_generation/upload_GNPS/final_files_GNPS/C28_alcohol_library_final.mgf")